In [1]:
%pip install diffusers transformers accelerate torch torchvision Pillow peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 7.6 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


# Imports

In [31]:
import os
from pathlib import Path
from PIL import Image
from typing import List

import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# For prompt: convert text to numbers, convert number to embeddings. 
#             UNet will use the embeddings to guide image creation.
from transformers import CLIPTokenizer, CLIPTextModel 

# AutoencoderKL (compress & reconstruct images): for compression and decompression btw pixel space and latent space.
# UNet2DConditionModel (predicts the noise): U-Net NN that takes (a noisy latent, timestep and text embedding) and predicts the noise to be removed.
# DDPMScheduler (adds noise - training): handles the noise process. How much noise to add at a certain timestemp. add_noise() - to use during training. 
# UniPCMultistepScheduler (removes noise - inference): the denoising solver - to be used during inference.
# StableDiffusionPipeline (wraps the above together): wraper for the 5 components (tokenizer, text encoder, VAE, UNet, scheduler) - to use at inference.

# Training: 
#     Image -> [AutoencoderKL] -> 
#     Latent -> [DDPMScheduler (add noise)] -> 
#     Noisy latent -> [UNet2DConditionModel] -> 
#     Predicted noise -> [loss]

# Inference:
#     Random noise -> [UniPCMultistepScheduler] -> 
#     Iterative denoising -> [UNet2DConditionModel] -> 
#     Clean latent -> [AutoencoderKL] -> 
#     Final Image
from diffusers import AutoencoderKL, DDPMScheduler, StableDiffusionPipeline, UNet2DConditionModel, UniPCMultistepScheduler
from diffusers.utils import make_image_grid

# LoraConfig: defines how LORA layers are inserted in the model. 
# get_peft_model: takes the pretrained model and injects the Lora adapters.

# The original U-Net weights are frozen -> 
# LoRA layers are added into attention -> 
# the optimizer updates only the LoRA and does not retrain the millions of parameters
from peft import LoraConfig, get_peft_model

# Dataset

In [48]:
class TrainDataset(Dataset):
    """
    Dataset class for training data

    """
    def __init__(self, folder_path: str):
        self.dataset_dir = Path(folder_path)
        #print("dataset init ", self.dataset_dir)
        self.images_paths = sorted(self.dataset_dir.glob("*.jpg"))  
        print("number of paths",len(self.images_paths))
        #print(self.images_paths[:10])
        self.transform = self.get_transform()

    def __len__(self) -> int:
        return len(self.images_paths)

    def __getitem__(self, index: int) -> torch.Tensor:
        img_path =  self.images_paths[index]
        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(img)
        return img_tensor

    @staticmethod
    def get_transform(
        img_size: int = 512, 
        norm_scalar: List[float] = [0.5] 
    ) -> transforms.Compose:
        t = transforms.Compose([
            transforms.Resize(img_size, interpolation=transforms.InterpolationMode.LANCZOS),
            transforms.CenterCrop(img_size),
            transforms.ToTensor(), 
            transforms.Normalize(norm_scalar, norm_scalar)
        ])
        return t

# Configurations

In [46]:
STYLE_IMAGES_PATH = "../00_input_data/images/00_datasets/vangogh/SaintRemy"
MODELS_OUTPUT_DIR = "models/vangogh"
OUTPUT_DATA_DIR = "output_data/images/vangogh"

In [11]:
MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"

In [13]:
STYLE_TYPE = "van Gogh"
PROMP_INPUT = f"stylization in {STYLE_TYPE} style"

In [14]:
LORA_RANK = 16 # 4-8: style is subtle, 16-32: style is stronger
LORA_ALPHA = 16 # scaling factor. Should be the same as rank
LORA_DROPOUT = 0.1 # make lora a little more noisy during training. This will allow to learn a cleaner style. 

In [44]:
TRAINABLE_LAYERS = ["to_q", "to_k", "to_v", "to_out.0"]
TRAINING_STEPS = 1000
TRAINING_STRENGTH = 0.3 # range btw 0.3 - 0.5 for best results (brushstrokes, colors, texture)
LEARNING_RATE = 1e-4
BATCH_SIZE = 1
NUMBER_OF_WORKERS = 4
GRADIENT_ACCUMULATION = 4
SAVE_FREQUENCY = int(TRAINING_STEPS / 5)
SEED = 42


# Device & data type

In [18]:
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Device: {device}; dtype: {dtype}")

Device: cuda; dtype: torch.bfloat16


# LoRA & Models 

In [41]:
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
prompt_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=dtype).to(device)

#AutoencoderKL, DDPMScheduler, StableDiffusionPipeline, UNet2DConditionModel, UniPCMultistepScheduler
autoencoder = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=dtype).to(device)
train_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet").to(device)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: stable-diffusion-v1-5/stable-diffusion-v1-5
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
# Freeze
prompt_encoder.requires_grad_(False)
autoencoder.requires_grad_(False)
unet.requires_grad_(False)

UNet2DConditionModel(
  (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): Linear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): Linear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): CrossAttnDownBlock2D(
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(
          (norm): GroupNorm(32, 320, eps=1e-06, affine=True)
          (proj_in): Conv2d(320, 320, kernel_size=(1, 1), stride=(1, 1))
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): Linear(in_features=320, out_features=320, bias=False)
                (to_k): Linear(in_features=320, out_features=320, bias=False)
                (to_v): Linear(in_features=320, out_fe

In [43]:
# Turn the normal stable diffusion unet to LoRA unet. Only some adapter layers are trainable
lora_configurations = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    init_lora_weights="gaussian",
    target_modules=TRAINABLE_LAYERS,
    bias="none"
)

lora_unet = get_peft_model(unet, lora_configurations) # inject lora weights and freezes original unet

lora_parameters = [param for param in unet.parameters() if param.requires_grad]
optimizer = optim.AdamW(lora_parameters, lr=LEARNING_RATE)

# Training

In [49]:
dataset = TrainDataset(STYLE_IMAGES_PATH)
data_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUMBER_OF_WORKERS,
    pin_memory=torch.cuda.is_available()
)


number of paths 138
